# Premier League 2024-25: Transfer Window Analysis

You have just joined the analytics team at a Premier League club. The transfer window opens in two weeks and the recruitment team is counting on you to help them make data-driven decisions.

Your job today: dig into the 2024-25 season stats and answer some real questions about players.

The data is real(almost). It covers every player from the 2024-25 Fantasy Premier League season and loads straight from the internet, so there is nothing to download.

Let's get into it.

---

## Step 1: Load the data and have a look around

Let's bring the dataset in and get a feel for what we are working with.

In [ ]:
import pandas as pd
import numpy as np







That's our dataset ready. Now let's actually use it.

---

## Step 2: Clean the player names

The raw data has `first_name` and `second_name` in separate columns. We need one column with the full name for reports and lookups.

While we are here, let's understand how strings actually work in Python.

### Combining strings

You can stick strings together using `+`. Let's try it:

Simple. Now let's do it across the whole dataset.

### Strings cannot be changed once created

This is one of the most important things to understand about strings in Python. Once a string exists, you cannot modify it. You can only create a new one.

Try this:

Any method you call on a string -- `.lower()`, `.strip()`, `.replace()` -- gives you back a new string. If you do not save it to a variable, the result is gone.

### Splitting strings

`.split()` breaks a string apart at a separator and gives you a list back.

`maxsplit` is useful when your separator appears more than once but you only want to cut at the first one. Here we pull the player name off the front and keep everything else intact.

---

## Step 3: Lists and tuples

Two data structures you will use constantly. Let's understand both properly.

### Nested lists

A nested list is a list that contains other lists. You have probably seen this already without realising -- when we select multiple columns from a DataFrame we use double brackets. More on that shortly.

First let's understand the indexing. You drill down layer by layer, one set of square brackets at a time.

Each set of brackets takes you one level deeper. Work through it step by step and it always makes sense.

### You have already seen this in pandas

When we select multiple columns from a DataFrame we write:

```python
data[['Player', 'Goals', 'assists']]
```

Those double brackets are not special syntax. The outer brackets are the indexing. The inner brackets are a list of column names. You are passing a list into the indexing operator -- same idea as nested lists.

### Tuples

Tuples look like lists but use round brackets. The key difference: you cannot change them after creating them.

That makes them the right choice for records that should stay fixed -- a player snapshot, a coordinate, a database row.

Lists are mutable -- you can change, add, and remove items freely. Tuples are immutable -- once created, that's it. Use tuples when the data should not change.

---

## Step 4: List comprehensions

A list comprehension is a shorter way to build a list. Instead of writing a for loop with `.append()`, you describe the list in one line.

Let's start with the long way so we understand what is happening:

In [ ]:
# The long way -- a for loop








In [ ]:
# The list comprehension way -- same result, one line










The pattern is always:

```python
[expression for item in iterable]
```

You can also add a condition to filter:

### Building a contribution score

For our analysis, we are going to say a goal is worth 2 points and an assist is worth 1. That is a choice we are making, not an official stat. You could change those weights and see how the rankings shift -- we will do exactly that when we get to functions.

For now let's calculate the contribution score for every player.

To do this across two columns at once, we need `zip`. Let's understand what it does first:

---

## Step 5: Filtering and sorting

This is where we start answering the recruitment team's actual questions.

### Sorting

`sort_values()` orders the rows. `ascending=False` puts the biggest values first.

### How filtering actually works

Before we filter, let's understand what is happening under the hood.

When you write a condition on a column, pandas checks every row and gives back True or False:

Now when you pass that into `data[...]`, pandas keeps only the rows where the value is True. That is all filtering is.

### Filtering on two conditions

Combine conditions with `&` (and) or `|` (or). Each condition must be in its own brackets -- Python needs this to evaluate them in the right order.

### Filter then aggregate

Filter to a subset, then calculate something on those rows. This pattern comes up constantly.

### Groupby

`groupby()` splits the data into separate groups and lets you calculate something for each group. Let's build it up slowly.

---

## Step 6: Write reusable functions

The analysis is growing. The same logic will run on different players, different subsets, next season's data. Time to write proper reusable functions.

### PEP 8

PEP 8 is Python's official style guide. It exists so any Python developer can read your code without decoding your personal formatting choices:

- One statement per line
- `snake_case` for function and variable names
- Spaces around operators: `goals * 2` not `goals*2`
- Docstring goes immediately inside the function, before any code

### Docstrings

A docstring is the description at the top of a function. Write one for every function you plan to share or reuse.

### Back to the goals vs assists

Let's write a function where we can change the weighting and see how the rankings shift. If a learner thinks assists should count the same, they can pass in `goal_weight=1` and `assist_weight=1` and see what happens to the top 10.

---

## Step 7: Lambda functions

Sometimes you need a function for a one-off job and giving it a full name feels like overkill. That's what lambdas are for.

### What is apply?

Before we get to lambdas, we need to understand `apply`. It runs a function on every value in a column -- one value at a time.

Let's see it with a named function first:

In [ ]:
data['Goals'].head()


In [ ]:
# A simple named function











So `apply` is just a way to run a function across an entire column. Now, instead of defining a named function, you can write the function inline using a lambda:

The pattern is always:

```python
lambda arguments: expression
```

Whatever comes after the colon is what gets returned. Lambdas are most useful when the function is short and you only need it once.

In [ ]:
# Tag every player with a category









---

## Step 8: Bundle the analysis into a class

We now have several functions all working on the same dataset. As the tool grows, passing the DataFrame into every function separately gets messy. A class lets us bundle the data and everything we want to do with it into one tidy package.

### What is a class?

A class is a blueprint. You define it once, then create objects from it. Each object has its own copy of the data and access to all the methods.

### __init__ and self

`__init__` runs automatically when you create an object -- it is the setup function. `self` refers to the specific object being created. When you write `self.data = dataframe`, you are storing the DataFrame on that object so every method can use it later.

### Instantiation

Instantiation is the act of creating an object from the blueprint. When you write `PlayerAnalyser(data)`, Python creates the object and runs `__init__` automatically.

Let's build the class step by step.

Good. The data is stored. Now let's add the first method.

### top_scorers

This method sorts the data by Goals descending, takes the top n rows, and returns selected columns. Let's walk through each step:

Now we understand what each line does, let's put it inside the class:

### position_summary

This method groups the data by position and calculates the average goals and assists for each group. We already know what groupby does from Step 5.

### best_value

This method finds the best points-per-cost players in a given position. Useful for the recruitment team who have a budget to work with.

---

## Step 9: Understanding distributions

The recruitment team has a shortlist of two forwards. Both average 10 goals a season. Which one do you sign?

You cannot answer that from the average alone. You need to know how consistent they are.

### Sample vs population variance

Now let's say you are scouting a striker and you only managed to watch 5 of his games this season. His goals in those 5 games: 0, 2, 1, 3, 0.

You calculate his average: 1.2 goals per game. You calculate the spread from that average. But here is the problem -- that average of 1.2 came from those same 5 games. It is already perfectly fitted to your small sample. So the spread you calculate will be slightly too tight, slightly too confident.

Dividing by n-1 instead of n is a small correction for that. It says: "we only watched 5 games out of 38, let's be a bit more honest about how well we actually know this player."

The more games you watched, the less the difference matters. Watch all 38 games and n-1 vs n is negligible.

`ddof` stands for "delta degrees of freedom". It is just the number you subtract from n before dividing. `ddof=1` means divide by n-1. That is all it is.

When in doubt, use `ddof=1`. One season is a sample of a player's whole career.

Now let's look at the spread in our actual dataset:

### The central limit theorem

Imagine you send 10 different scouts to watch different forwards across the league. Each scout watches a different set of games and comes back with an average goals per game for their players.

Those 10 averages will not all be the same. But here is the interesting thing: if you plot them, they will form a roughly normal distribution -- a bell curve centred around the true average, even if the raw game-by-game data is messy and all over the place.

That is the central limit theorem. It says: take enough samples, calculate the mean of each, and those means will cluster into a normal distribution regardless of what the original data looks like.

Why does this matter? Most statistical tests assume the data is normally distributed. Goals scored is not -- most players score zero, a few score a lot. But the CLT is what allows us to still run those tests, because it is the distribution of means that needs to be normal, not the raw data itself.

---

## Step 10: Visualise the data

The board presentation needs charts. Let's build up to the most informative one gradually, understanding each chart along the way.

### Bar chart

A bar chart shows one value per group -- usually a total or an average. Simple and familiar. Let's start here.

The bar chart tells you: on average, which position scores most.

But it hides everything else. Two positions could have the same average but completely different spreads. One might have a few superstars pulling the average up while most players score nothing. The bar chart cannot show you that.

### Box plot

A box plot shows more than just the average. It shows you:
- The line in the middle: the median (middle value)
- The box: the middle 50% of players (between the 25th and 75th percentile)
- The lines extending out (whiskers): the range of typical values
- The dots beyond the whiskers: outliers

Better. Now you can see the spread within each position, not just the average. You can see that forwards have a much wider range than goalkeepers.

But you still cannot see where the data is dense. Are most forwards scoring around 5 goals with a few at 20, or is it spread evenly? The box plot cannot tell you that.

### Violin plot

A violin plot combines the box plot with a density estimate. The width of the shape at any point tells you how many players have values around there. Wide means many players. Narrow means few.

Now you can see the full picture:
- The wide bottom of each violin shows where most players cluster (near zero for everyone)
- The thin tail at the top shows the rare high scorers
- Forwards have the longest tail -- the biggest outliers come from that position

A few things to watch out for with any chart:
- `x` should be the groups (categories), `y` should be the numeric value -- swapping them produces a chart that makes no sense
- Make sure you are using the right chart type for what you are trying to show
- Forgetting `plt.show()` in a script means nothing displays. Jupyter shows plots automatically, scripts do not.

---

## Step 11: Is the difference real?

The recruitment team is debating: do forwards genuinely outscore midfielders, or does it just look that way because of a few big names?

You could compare averages. But averages can be misleading. A few outliers like Salah or Haaland could pull one group's average up while most players in that group score very little. How do we know if the difference is real?

### The question a t-test answers

A t-test asks: how confident are we that the difference between two groups is real and not just a fluke of this particular season's data?

Let's think about it with a simple example first. Say you have two small groups:

- Group A: forwards from your local Sunday league: [2, 3, 2, 4, 3]
- Group B: midfielders from the same league: [1, 2, 1, 2, 1]

Group A averages 2.8, Group B averages 1.4. There is a difference. But is it real, or would it disappear if you watched different games?

### The t-statistic

The t-statistic measures how big the difference is relative to the natural variation within each group.

Think of it this way:
- Forwards average 5 goals, midfielders average 3. Difference of 2.
- But if forwards' individual tallies are all over the place -- some score 0, some score 20 -- that 2 goal difference does not mean much.
- If almost every forward scores between 4 and 6, then a 2 goal difference is significant.

So the t-statistic is roughly: difference between the two means divided by the spread within each group.

Big t-statistic: the difference is large relative to the noise. Likely real.
Small t-statistic: the difference could easily be noise.

### The p-value

The p-value converts the t-statistic into a probability. Specifically: what is the probability of seeing a difference this large if there were actually no real difference between the two groups?

- Small p-value (below 0.05): the difference is unlikely to be a fluke. We call it statistically significant.
- Large p-value (above 0.05): we cannot rule out that this is just noise.

The 0.05 threshold is a convention, not a law. It means: we are willing to accept a 5% chance of being wrong.

Let's see it in code with a simple made-up example first:

That p-value will be tiny because the two groups are clearly different. Now let's try with two groups that are more similar:

Now the p-value is large because the groups overlap so much. Now let's answer the real question: